In [2]:
!mkdir -p drive
!google-drive-ocamlfuse drive

/bin/bash: google-drive-ocamlfuse: command not found


In [0]:
import sys
sys.path.insert(0, 'drive/')

In [4]:
!pip install -q keras
!pip install -q tensorflow
!pip install -q numpy
!pip install -q pandas
!pip install -q nltk
!pip install -U -q PyDrive
!pip install -U -q sumeval
!pip install -U -q ProgressBar
!pip install -U -q networkx
!pip install -U -q sumy






     |████████████████████████████████| 993kB 2.8MB/s 
     |████████████████████████████████| 51kB 2.0MB/s 
     |████████████████████████████████| 92kB 3.5MB/s 
     |████████████████████████████████| 10.5MB 8.5MB/s 


In [0]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# 1. Authenticate and create the PyDrive client.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

# 2. Load a file by ID and create local file.
downloaded = drive.CreateFile({'id':'1RZ7L1ToT_f5cj73U26Dbi7sIm2V4xznF'}) # replace fileid with Id of file you want to access
downloaded.GetContentFile('wikihowAll.csv') # now you can use export.csv 

downloaded2 = drive.CreateFile({'id':'1_M0ya_yrrNwTEEPXjXycDM73KM8CR1Ln'}) # replace fileid with Id of file you want to access
downloaded2.GetContentFile('glove.6B.100d.txt') # now you can use export.csv 

In [6]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize 
from sklearn.utils.extmath import randomized_svd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import networkx
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
import heapq
from sumeval.metrics.rouge import RougeCalculator
from nltk.translate.bleu_score import sentence_bleu
from progressbar import ProgressBar

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Unzipping corpora/wordnet.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [0]:
wikiHow = pd.read_csv("wikihowAll.csv")

In [8]:
wikiHow.shape

(215365, 3)

In [9]:
wikiHow.head()

,headline,title,text
0,"\nKeep related supplies in the same area.,\nMa...",How to Be an Organized Artist1,"If you're a photographer, keep all the necess..."
1,\nCreate a sketch in the NeoPopRealist manner ...,How to Create a Neopoprealist Art Work,See the image for how this drawing develops s...
2,"\nGet a bachelor’s degree.,\nEnroll in a studi...",How to Be a Visual Effects Artist1,It is possible to become a VFX artist without...
3,\nStart with some experience or interest in ar...,How to Become an Art Investor,The best art investors do their research on t...
4,"\nKeep your reference materials, sketches, art...",How to Be an Organized Artist2,"As you start planning for a project or work, ..."


In [10]:
wikiHow.isnull().sum()

headline     818
title          1
text        1071
dtype: int64

In [0]:
wikiHow = wikiHow.dropna()
wikiHow = wikiHow.drop(['title'], 1)
wikiHow['text']=wikiHow['text'].fillna("")
wikiHow['headline']=wikiHow['headline'].fillna("")
wikiHow = wikiHow.reset_index(drop=True)

In [12]:
wikiHow.head()

,headline,text
0,"\nKeep related supplies in the same area.,\nMa...","If you're a photographer, keep all the necess..."
1,\nCreate a sketch in the NeoPopRealist manner ...,See the image for how this drawing develops s...
2,"\nGet a bachelor’s degree.,\nEnroll in a studi...",It is possible to become a VFX artist without...
3,\nStart with some experience or interest in ar...,The best art investors do their research on t...
4,"\nKeep your reference materials, sketches, art...","As you start planning for a project or work, ..."


In [13]:
wikiHow.tail()

,headline,text
214289,\nConsider changing the spelling of your name....,"If you have a name that you like, you might f..."
214290,"\nTry out your name.,\nDon’t legally change yo...",Your name might sound great to you when you s...
214291,"\nUnderstand the process of relief printing.,\...",Relief printing is the oldest and most tradit...
214292,\nUnderstand the process of intaglio printing....,"Intaglio is Italian for ""incis­ing,"" and corr..."
214293,\nUnderstand the different varieties of lithog...,Lithography is a big term often used to refer...


In [14]:
wikiHow.text[7440]

';\n,,, In and then out.\n\n,,,,'

In [15]:
wikiHow.headline[7440]

'\nPlace the ball where down to the ground and face the ball valve (the hard part of the ball) toward your position.,\nTake about 3 big step back to give you more room to work with, it also help when you strike the ball.\n\n,\nPlace yourself behind the ball, and pick the direction that you want the ball to go to.\n\n,\nTake a deep breath before taking the shot.,\nThen run up to the ball and place your supporting foot away from the ball about a couple inches.\n\n,\nUse all the power you had toward your kicking foot, and your upper body should leaning back a little bit to lift the ball up and over the wall.\n\n,\nKick the ball with your lace or your front foot because this give you more power and accurate toward your shot.\n\n,\nFinally hit the bottom of the ball fast and lower your body as you hit the ball.\n\n'

In [0]:
contractions = { 
"ain't": "am not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he's": "he is",
"how'd": "how did",
"how'll": "how will",
"how's": "how is",
"i'd": "i would",
"i'll": "i will",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'll": "it will",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"must've": "must have",
"mustn't": "must not",
"needn't": "need not",
"oughtn't": "ought not",
"shan't": "shall not",
"sha'n't": "shall not",
"she'd": "she would",
"she'll": "she will",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"that'd": "that would",
"that's": "that is",
"there'd": "there had",
"there's": "there is",
"they'd": "they would",
"they'll": "they will",
"they're": "they are",
"they've": "they have",
"wasn't": "was not",
"we'd": "we would",
"we'll": "we will",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"where'd": "where did",
"where's": "where is",
"who'll": "who will",
"who's": "who is",
"won't": "will not",
"wouldn't": "would not",
"you'd": "you would",
"you'll": "you will",
"you're": "you are"
}

In [0]:
def clean_text(text,contradictions = True):
    # Convert words to lower case
    
    if type(text) is str:
        text = text.lower()
    
    # Replace contractions with their longer forms 
    if contradictions:
        text = text.split()
        new_text = []
        for word in text:
            if word in contractions:
                new_text.append(contractions[word])
            else:
                new_text.append(word)
        text = " ".join(new_text)
        
    if re.search(r'(\w+)', text) is None:
      #print(text)      
      text = ''
    if re.search(r'^(\w+)', text) is None:
      #print(text)      
      text = ''
      
      
    
    # Format words and remove unwanted characters
    text = re.sub(r'\<a href', ' ', text)
    text = re.sub(r'http\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'&amp;', '', text) 
    #text = re.sub(r'[_"\-;%()|+&=*%.,!?:#$@\[\]/]', ' ', text)
    text = re.sub(r'<br\s*\/?>', '', text)
    text = re.sub(r'<br />', ' ', text)
    text = re.sub(r'</a>', ' ', text)
    text = re.sub(r'=', ' ', text)
    text = re.sub(r'"', ' ', text)
    text = re.sub(r'  ', ' ', text)
    text = re.sub(r' ''',' ', text, flags=re.MULTILINE)
    #text = re.sub(r'[,.]+$', '', text)
    
    #text = re.sub(r'.+a n[^a-z]+''t.+','', text, flags=re.MULTILINE)
    text = re.sub(r'^([a-z]\s)+[a-z]$','', text, flags=re.MULTILINE)
    #text = re.sub(r'\'', ' ', text)

    return text

In [18]:
clean_summaries = []

for summary in wikiHow.headline:
    clean_summaries.append(clean_text(summary,contradictions = True))
print("Summaries are complete.")



Summaries are complete.


In [19]:
clean_texts = []
for text in wikiHow.text:
    clean_texts.append(clean_text(text,contradictions = True))
print("Texts are complete.")



Texts are complete.


In [20]:
clean_texts[15]

"whether you are teaching a skill, delivering information or increasing awareness, outline the goals of your workshop. what do you want your workshop participants to learn? this analysis may result in a list of specific skills you will be teaching, concrete topics you will cover, or simply a feeling you will inspire in your participants. think carefully about what you want to accomplish and why it is important.some examples of workshop objectives include: learn how to write a persuasive cover letter. learn how to break bad news to a patient. learn 5 techniques to get a reluctant student to talk in class. learn how to create an effective powerpoint presentation.; , will the workshop participants know one another or are they strangers? will they come in with knowledge about your topic or will they be completely unfamiliar with it? are they choosing to attend your workshop or is it a requirement for their job training? answers to all of these questions will affect how you organize your wo

In [21]:
clean_summaries[15]

'define the workshop objective., decide who your audience is., schedule your workshop for the morning or early afternoon., publicize your workshop., recruit 8-15 participants for your workshop., prepare your participants for the workshop., prioritize your goals for the workshop., prepare a variety of teaching aids., prepare paper handouts., arrange your audio-visual materials., organize your computer-based materials., recruit experts, speakers, and assistants., decide on your group activities., leave time for breaks., resist cramming., secure catering., arrive early., set up all equipment before participants arrive., arrange the chairs in advance., distribute materials., greet participants as they arrive., introduce yourself and the workshop., begin icebreakers., execute your lesson plan., be flexible., use interactive exercises to reinforce information., do not talk too much., stick to your scheduled breaks., switch up activities every 20-30 minutes., lighten the mood., maintain a res

In [0]:
def tokenizer(text):
    document = text.strip()
    sentences = nltk.sent_tokenize(document)
    sentences = [sentence.strip() for sentence in sentences]
    #text = ' '.join(sentences)
    return sentences




In [23]:
sentences = tokenizer(clean_texts[713])
sentences


['you should not change yourself for a guy because then he will like the fake you and not the real you, and that is not good.',
 'that is taking the wrong direction.',
 'do not go there.',
 'do whatever you feel like doing and act however you feel like acting.',
 'dart if you feel like.',
 'if you usually pick your nose, pick your nose!',
 'wear what you like, say what you like, act how you like.',
 'do whatever you feel like.',
 'if you are genuinely a quiet, calm, serious sort of person be like that.',
 'if you are really kinda crazy, be crazy!',
 'if he does not like you for who you are, he is not worth it.',
 '; , he would rather see your face and real body then all that fake glob on your face and clothes that show your body off.',
 'its okay to put on some make-up, maybe even a little blush or something else (whatever you want) but - do not use what you feel is too much.',
 'it is for your own good.',
 'he will not see the real you if you put fake stuff on you when your really bea

In [0]:
def lemmatizer(sentences):
    lemmatizer = nltk.WordNetLemmatizer()
    for i in range(len(sentences)):
        #print(sentences[i])
        words = nltk.word_tokenize(sentences[i])
        words = [lemmatizer.lemmatize(word) for word in words]
        sentences[i] = ' '.join(words)
    return sentences

In [0]:
def stemmer(sentences):
    stemmer = PorterStemmer()
    for i in range(len(sentences)):
        #print(sentences[i])
        words = nltk.word_tokenize(sentences[i])
        words = [stemmer.stem(word) for word in words]
        sentences[i] = ' '.join(words)
    return sentences

In [0]:
def stopwords(sentences):
    stopword_list = nltk.corpus.stopwords.words('english')
    word_token_list = []
    filtered_sentence_item = [] 
    filtered_sentence = []
    
    for sentence in sentences:
        word_tokens = word_tokenize(sentence)
        word_token_list.append(word_tokens)
    
    for word_tokens in word_token_list:
        for word in word_tokens:
            if(word not in stopword_list):
                filtered_sentence_item.append(word)
                #print(filtered_sentence_item)
        filtered_sentence_item = ' '.join(filtered_sentence_item)
        filtered_sentence.append(filtered_sentence_item)
        #print(filtered_sentence_item)
        filtered_sentence_item = [] 
            
    return filtered_sentence

In [0]:
def vectorizer(Tdidf=True):
    if Tdidf:
        vectorizer = TfidfVectorizer(min_df=1,ngram_range=(1, 1))
    else:
        vectorizer = CountVectorizer(binary=False, min_df=1,ngram_range=(1, 1))
    return vectorizer

In [0]:
def td_matrix(sentences,vectorizer):
    feature_matrix = vectorizer.fit_transform(sentences)
    td_matrix = feature_matrix.transpose()
    #td_matrix = td_matrix.multiply(td_matrix > 0)
    return td_matrix

In [0]:
def svd(td_matrix):
    u, s, vt = randomized_svd(td_matrix,
                              n_components=1,
                              n_iter=100,
                              random_state=None)
    return u,s,vt

In [0]:
def CalculateRougeScores(refrence_summary,model_summary,scoring = False):
    rouge = RougeCalculator(stopwords=True, lang="en")
    rouge_1 = rouge.rouge_n(
            summary=model_summary,
            references=refrence_summary,
            n=1)
    rouge_2 = rouge.rouge_n(
            summary=model_summary,
            references=[refrence_summary],
            n=2)
    rouge_L = rouge.rouge_l(
            summary=model_summary,
            references=[refrence_summary])
    if (scoring == False):
        print("ROUGE-1: {}, ROUGE-2: {}, ROUGE-L: {}".format(
            rouge_1, rouge_2, rouge_L
        ).replace(", ", "\n"))
    
    return rouge_1,rouge_2,rouge_L

In [0]:
def CalculateBleuScores(refrence_summary,model_summary,scoring = False):
    refrence_summary_list = []
    rs =nltk.word_tokenize(refrence_summary)
    #print(r)
    refrence_summary_list.append(rs)
#     print(refrence_summary_list)
    model_summary = nltk.word_tokenize(model_summary)

#     print(model_summary)
#     a = [['good', 'quality', 'dog', 'food']]
#     b = ['a', 'good', 'dog', 'food']
    
    score = sentence_bleu(refrence_summary_list, model_summary,weights=(1, 0, 0, 0))
    if(scoring == False):
        print("BLEU Score: {} ".format(
            score,).replace(", ", "\n"))
    return score

In [0]:
def LsaSummarizer(text,summary,scoring = False):
    
    rouge_1 = 0
    rouge_2 = 0
    rouge_L = 0
    bleu = 0
    
    if  text and text.strip():
        sentences = tokenizer(text)
        original_sentences = sentences
        sentences = lemmatizer(sentences)
        #sentences = stemmer(sentences)
        sentences = stopwords(sentences)
        vectorizerVal = vectorizer(Tdidf=True)
        td_matrixVal = td_matrix(sentences,vectorizerVal)
        #print(td_matrixVal)
        u,s,vt = svd(td_matrixVal)
        #print(vt.shape)
        num_sentences = 1

        salience_scores = np.sqrt(np.dot(np.square(s), np.square(vt)))
        #print(len(salience_scores))
        top_sentence_indices = salience_scores.argsort()[-num_sentences:][::-1]
        top_sentence_indices.sort()
        #print(original_sentences)
        #print(top_sentence_indices)
        model_summary = ''
        if(scoring == False):
            print('Original Text: ' + text + '\n')
            print('Human Summary: ' + summary + '\n')      

            for index in top_sentence_indices:
                print ('Lsa Summarizer: ' + original_sentences[index] + '\n')
                model_summary = original_sentences[index]
        else:
             for index in top_sentence_indices:
                model_summary = original_sentences[index]

        rouge_1,rouge_2,rouge_L = CalculateRougeScores(summary,model_summary,scoring)
        bleu = CalculateBleuScores(summary,model_summary,scoring)
        #print(rouge_1,rouge_2,rouge_L,bleu)
        return rouge_1,rouge_2,rouge_L,bleu
    else:
        #print(rouge_1,rouge_2,rouge_L,bleu)
        return rouge_1,rouge_2,rouge_L,bleu
        
    
    

In [33]:
LsaSummarizer(clean_texts[7440],clean_summaries[7440])


(0, 0, 0, 0)

In [0]:
def similarity_matrix(vectorizer):
    feature_matrix = vectorizer.fit_transform(sentences)
    similarity_matrix = (feature_matrix * feature_matrix.T)
    return similarity_matrix

In [0]:
def similarity_graph(similarity_matrix):
    similarity_graph = networkx.from_scipy_sparse_matrix(similarity_matrix)
    scores = networkx.pagerank(similarity_graph)
    networkx.draw_networkx(similarity_graph)
    return scores

In [0]:
def TextRankSummarizer(text,summary,scoring = False):
    
    rouge_1 = 0
    rouge_2 = 0
    rouge_L = 0
    bleu = 0
    
    if  text and text.strip():
        sentences = tokenizer(text)
        original_sentences = sentences
        sentences = lemmatizer(sentences)
        sentences = stopwords(sentences)
        vectorizerVal = vectorizer(Tdidf=True)
        similarityMatrixVal = similarity_matrix(vectorizerVal)
        scores = similarity_graph(similarityMatrixVal)
#         print(scores)
        num_sentences = 1
        ranked_sentences = sorted(((score, index)
                               for index, score
                               in scores.items()),
                              reverse=True)

        top_sentence_indices = [ranked_sentences[index][1] for index in range(num_sentences)]
        top_sentence_indices.sort()
        model_summary = ''
        if(scoring == False):             
            print('Original Text: ' + text + '\n')
            print('Human Summary: ' + summary + '\n')
            for index in top_sentence_indices:
                if(len(original_sentences)> index):
                  print ('TextRank Summarizer: ' + original_sentences[index] + '\n')
                  model_summary = original_sentences[index]
        else:
                for index in top_sentence_indices:
                    if(len(original_sentences)> index):
                        model_summary = original_sentences[index]            

        rouge_1,rouge_2,rouge_L = CalculateRougeScores(summary,model_summary,scoring)
        bleu = CalculateBleuScores(summary,model_summary,scoring)
        return rouge_1,rouge_2,rouge_L,bleu
    else:
        return rouge_1,rouge_2,rouge_L,bleu
   
    
    
    

In [37]:
TextRankSummarizer(clean_texts[7440],clean_summaries[7440])

(0, 0, 0, 0)

In [0]:
def LexRankSummarizerSumy(text,humanSummary,scoring = False):
    
    rouge_1 = 0
    rouge_2 = 0
    rouge_L = 0
    bleu = 0
    
    if  text and text.strip():
        summarizer = LexRankSummarizer()
        parser = PlaintextParser.from_string(text,Tokenizer("english"))
        summary = summarizer(parser.document, 1)

        model_summary = ''
        if(scoring == False):
            print('Original Text: ' + text + '\n')
            print('Human Summary: ' + humanSummary + '\n')
            for sentence in summary:
                print('LexRank Summarizer: ' + str(sentence)+ '\n')
                model_summary = str(sentence)
        else:
            for sentence in summary:
                model_summary = str(sentence)            

        rouge_1,rouge_2,rouge_L = CalculateRougeScores(humanSummary,model_summary,scoring)
        bleu = CalculateBleuScores(humanSummary,model_summary,scoring)
        
        return rouge_1,rouge_2,rouge_L,bleu
    else:
        return rouge_1,rouge_2,rouge_L,bleu
      


In [39]:
LexRankSummarizerSumy(clean_texts[7440],clean_summaries[7440])

(0, 0, 0, 0)

In [0]:
def get_keywords(sentences):
    
    filtered_text = ' '.join(sentences)
    word_list = re.findall(r'\w+', filtered_text)    
    count_dict = {}
    
    for word in word_list:
        count_dict.setdefault(word, 0)
        count_dict[word] += 1

    keywords = set()
    
    for word , cnt in count_dict.items():
        word_percentage = count_dict[word] * 1.0 / len(word_list)
        if word_percentage <= 0.5 and word_percentage >= 0.001:
            keywords.add(word)
    
    return keywords

In [0]:
def calculate_sentence_weight(text,keywords):
    
    keywords = keywords
    sentence_weight = []
    sentence_weight_val = 0
    filtered_sentence_list = text.split('.')
    
    for sentence in filtered_sentence_list:
        sentence_list = sentence.split(' ')
        window_start = 0
        window_end = -1
        # calculating window start
        for i in range(len(sentence_list)):
            if sentence_list[i] in keywords:
                window_start = i
                break
        # calculating window end
        for i in range(len(sentence_list) - 1, 0, -1):
            if sentence_list[i] in keywords:
                window_end = i
                break

        window_size = window_end - window_start + 1
        # calculate number of keywords
        keywords_cnt = 0

        for word in sentence_list:
            if word in keywords:
                keywords_cnt += 1

        if window_start > window_end:
            sentence_weight_val = 0
        else:
            sentence_weight_val = keywords_cnt*keywords_cnt *1.0 / window_size
        
        sentence_weight.append((sentence_weight_val,sentence))
        sentence_weight.sort(reverse = True)
    
    return sentence_weight,sentence_list

In [0]:
def LuhnSummarizer(text,summary,scoring = False):

    rouge_1 = 0
    rouge_2 = 0
    rouge_L = 0
    bleu = 0
    
    if  text and text.strip():
        sentences = tokenizer(text)
        sentences = lemmatizer(sentences)
        sentences = stopwords(sentences)
        keywords = get_keywords(sentences)
        sentence_weight,sentence_list = calculate_sentence_weight(text,keywords)  

        #print sentence_weight
        ret_list = []
        ret_cnt = min(1 , len(sentence_list))

        for i in range (ret_cnt):
            ret_list.append(sentence_weight[i][1])

        model_summary = ''
        if(scoring == False):
            print('Original Text: ' + text + '\n')
            print('Human Summary: ' + summary + '\n')
            for sentence in ret_list:
                print ('Luhn Summarizer: ' + sentence + '\n')
                model_summary = sentence
        else:
            for sentence in ret_list:
                model_summary = sentence            

        rouge_1,rouge_2,rouge_L = CalculateRougeScores(summary,model_summary,scoring)
        bleu = CalculateBleuScores(summary,model_summary,scoring)
        return rouge_1,rouge_2,rouge_L,bleu
    else:
        return rouge_1,rouge_2,rouge_L,bleu

    
    

In [43]:
LuhnSummarizer(clean_texts[60],clean_summaries[60])

Original Text: before you approach your crush, you want to ease into the dance and get comfortable. talking with your friends is a great way to spend the first part of the dance. it also can show your crush if they are watching that you are a social and friendly person.; , even if you are nervous at the prospect of talking to your crush, try to have confident body language from the beginning of the dance to the end of the night. stand up straight, smile, and don’t try to hide yourself. people naturally attracted to confident and positive body language, and if your crush sees you looking uncomfortable or worried they might not want to dance.even if you feel nervous, smiling and standing up straight can actually trick your body into feeling calmer and more confident. that’s why even if you are nervous, you should fake it till you make it. try not to cross your arms or use other closed body language that hides your body. also try not to hang your head; instead keep it up with your chin le

/usr/local/lib/python3.6/dist-packages/nltk/translate/bleu_score.py:490: UserWarning: 
Corpus/Sentence contains 0 counts of 3-gram overlaps.
BLEU scores might be undesirable; use SmoothingFunction().
  warnings.warn(_msg)


(0.45454545454545453,
 0.09999999999999999,
 0.3636363636363636,
 0.23076923076923078)

In [0]:
def CountBasedSummarizer(text,summary,scoring = False):
    
    rouge_1 = 0
    rouge_2 = 0
    rouge_L = 0
    bleu = 0
    
    if  text and text.strip():
        sentences = tokenizer(text)
        original_sentences = sentences
        sentences = lemmatizer(sentences)
        #print(sentences)
        sentences = stopwords(sentences)  

        # Textte geçen her kelimenin sayısını buluyoruz.
        word2count = {}
        for sentence in sentences:  
            sentence = re.sub(r'\W', ' ', sentence)
            sentence = re.sub(r'\s+', ' ', sentence)
            for word in nltk.word_tokenize(sentence):
                    if word not in word2count.keys():
                        word2count[word] = 1
                    else:
                        word2count[word] += 1

        # Selecting best 50 features
        freq_words = heapq.nlargest(50, word2count, key=word2count.get)   
        # Textte en fazla geçen kelimenin sayısını diğer kelimelerin geçme sayılarına bölerek ağırlıkları buluyoruz.
        max_count = max(word2count.values())

        for key in word2count.keys():
            word2count[key] = word2count[key] / max_count
        # textteki cümleler içerisinde 25 kelimeden az olan sentence score  dictionarysini dolduruyoruz.

        sent2score = {}

        for sentence in original_sentences:  
            for word in nltk.word_tokenize(sentence.lower()):              
                if word in word2count.keys():
                    if len(sentence.split(' ')) < 50:
                        if sentence not in sent2score.keys():
                            sent2score[sentence] = word2count[word]
                        else:
                            sent2score[sentence] += word2count[word]
        #print(sent2score)
    #     return
        # heapq.nlargest dictionary deki scoreu en yüksek cümleyi getirir.
        best_sentences = heapq.nlargest(1, sent2score, key=sent2score.get)
        #print(best_sentences)
        model_summary = ''
        if(scoring == False):
            print('Original Text: ' + text + '\n')
            print('Human Summary: ' + summary + '\n')
            for sentence in best_sentences:
                print ('Count Based Summarizer: ' + sentence + '\n')
                model_summary = sentence
        else:
            for sentence in best_sentences:
                model_summary = sentence
        rouge_1,rouge_2,rouge_L = CalculateRougeScores(summary,model_summary,scoring)
        bleu = CalculateBleuScores(summary,model_summary,scoring) 
        
        return rouge_1,rouge_2,rouge_L,bleu
    else:
        return rouge_1,rouge_2,rouge_L,bleu
    

In [0]:
def Scoring():    
    pbar = ProgressBar()
    rouge_1_lsa = 0
    rouge_2_lsa = 0
    rouge_L_lsa = 0
    bleu_lsa = 0
    
    rouge_1_textrank = 0
    rouge_2_textrank = 0
    rouge_L_textrank = 0
    bleu_textrank = 0
    
    rouge_1_lexrank = 0
    rouge_2_lexrank = 0
    rouge_L_lexrank = 0
    bleu_lexrank = 0
    
    rouge_1_luhn = 0
    rouge_2_luhn = 0
    rouge_L_luhn = 0
    bleu_luhn = 0
    
    rouge_1_countbased = 0
    rouge_2_countbased = 0
    rouge_L_countbased = 0
    bleu_countbased = 0
    
    for i in pbar(range(len(clean_texts))):
        print(i)
        rouge_1,rouge_2,rouge_L,bleu = LsaSummarizer(clean_texts[i],clean_summaries[i],True)
        rouge_1_lsa = rouge_1_lsa + rouge_1
        rouge_2_lsa = rouge_2_lsa + rouge_2
        rouge_L_lsa = rouge_L_lsa + rouge_L
        bleu_lsa = bleu_lsa + bleu
        
        rouge_1,rouge_2,rouge_L,bleu = TextRankSummarizer(clean_texts[i],clean_summaries[i],True)
        rouge_1_textrank = rouge_1_textrank + rouge_1
        rouge_2_textrank = rouge_2_textrank + rouge_2
        rouge_L_textrank = rouge_L_textrank + rouge_L
        bleu_textrank = bleu_textrank + bleu

        rouge_1,rouge_2,rouge_L,bleu = LexRankSummarizerSumy(clean_texts[i],clean_summaries[i],True)
        rouge_1_lexrank = rouge_1_lexrank + rouge_1
        rouge_2_lexrank = rouge_2_lexrank + rouge_2
        rouge_L_lexrank = rouge_L_lexrank + rouge_L
        bleu_lexrank = bleu_lexrank + bleu
        
        rouge_1,rouge_2,rouge_L,bleu = LuhnSummarizer(clean_texts[i],clean_summaries[i],True)
        rouge_1_luhn = rouge_1_luhn + rouge_1
        rouge_2_luhn = rouge_2_luhn + rouge_2
        rouge_L_luhn = rouge_L_luhn + rouge_L
        bleu_luhn = bleu_luhn + bleu
        
        rouge_1,rouge_2,rouge_L,bleu = CountBasedSummarizer(clean_texts[i],clean_summaries[i],True)
        rouge_1_countbased = rouge_1_countbased + rouge_1
        rouge_2_countbased = rouge_2_countbased + rouge_2
        rouge_L_countbased = rouge_L_countbased + rouge_L
        bleu_countbased = bleu_countbased + bleu
        
    
    rouge_1_lsa_avg_score = rouge_1_lsa / len(clean_texts)
    rouge_2_lsa_avg_score = rouge_2_lsa / len(clean_texts)
    rouge_L_lsa_avg_score = rouge_L_lsa / len(clean_texts)
    bleu_lsa_avg_score = bleu_lsa / len(clean_texts)
    
    rouge_1_textrank_avg_score = rouge_1_textrank / len(clean_texts)
    rouge_2_textrank_avg_score = rouge_2_textrank / len(clean_texts)
    rouge_L_textrank_avg_score = rouge_L_textrank / len(clean_texts)
    bleu_textrank_avg_score = bleu_textrank / len(clean_texts)
    
    rouge_1_lexrank_avg_score = rouge_1_lexrank / len(clean_texts)
    rouge_2_lexrank_avg_score = rouge_2_lexrank / len(clean_texts)
    rouge_L_lexrank_avg_score = rouge_L_lexrank / len(clean_texts)
    bleu_lexrank_avg_score = bleu_lexrank / len(clean_texts)

    rouge_1_luhn_avg_score = rouge_1_luhn / len(clean_texts)
    rouge_2_luhn_avg_score = rouge_2_luhn / len(clean_texts)
    rouge_L_luhn_avg_score = rouge_L_luhn / len(clean_texts)
    bleu_luhn_avg_score = bleu_luhn / len(clean_texts)
    
    rouge_1_countbased_avg_score = rouge_1_countbased / len(clean_texts)
    rouge_2_countbased_avg_score = rouge_2_countbased / len(clean_texts)
    rouge_L_countbased_avg_score = rouge_L_countbased / len(clean_texts)
    bleu_countbased_avg_score = bleu_countbased / len(clean_texts)
    
    print("ROUGE-1-LSA: {}, ROUGE-2-LSA: {}, ROUGE-L-LSA: {}, BLEU-LSA: {}".
          format(rouge_1_lsa_avg_score,rouge_2_lsa_avg_score,rouge_L_lsa_avg_score,bleu_lsa_avg_score).replace(", ", "\n"))
    
    print("ROUGE-1-TextRank: {}, ROUGE-2-TextRank: {}, ROUGE-L-TextRank: {}, BLEU-TextRank: {}".
          format(rouge_1_textrank_avg_score,rouge_2_textrank_avg_score,rouge_L_textrank_avg_score,bleu_textrank_avg_score).replace(", ", "\n"))
    
    print("ROUGE-1-LexRank: {}, ROUGE-2-LexRank: {}, ROUGE-L-LexRank: {}, BLEU-LexRank: {}".
          format(rouge_1_lexrank_avg_score,rouge_2_lexrank_avg_score,rouge_L_lexrank_avg_score,bleu_lexrank_avg_score).replace(", ", "\n"))

    print("ROUGE-1-Luhn: {}, ROUGE-2-Luhn: {}, ROUGE-L-Luhn: {}, BLEU-Luhn: {}".
          format(rouge_1_luhn_avg_score,rouge_2_luhn_avg_score,rouge_L_luhn_avg_score,bleu_luhn_avg_score).replace(", ", "\n"))

    print("ROUGE-1-CountBased: {}, ROUGE-2-CountBased: {}, ROUGE-L-CountBased: {}, BLEU-CountBased: {}".
          format(rouge_1_countbased_avg_score,rouge_2_countbased_avg_score,rouge_L_countbased_avg_score,bleu_countbased_avg_score).replace(", ", "\n"))
    
    #print(rouge_1_lsa_avg_score,rouge_2_lsa_avg_score,rouge_L_lsa_avg_score,bleu_lsa_avg_score) 
    #return rouge_1_lsa_avg_score,rouge_2_lsa_avg_score,rouge_L_lsa_avg_score,bleu_lsa_avg_score
    
    
    

In [0]:
Scoring()

In [109]:
CountBasedSummarizer(clean_texts[486],clean_summaries[486])

ValueError: ignored

In [108]:
LsaSummarizer(clean_texts[486],clean_summaries[486])

ValueError: ignored

In [0]:
TextRankSummarizer(clean_texts[75],clean_summaries[75])

In [0]:
LexRankSummarizerSumy(clean_texts[75],clean_summaries[75])

In [0]:
LuhnSummarizer(clean_texts[75],clean_summaries[75])